# Diabetify CF Model Experiment

Notebook ini dipakai untuk eksperimen inferensi model bersama yang digunakan oleh `diabetify-cf`.

Cakupan notebook:
- load `xg_model.pkl` dan `x_columns.pkl`
- load `artifacts/reference/reference_data.parquet` bila tersedia
- prediksi baseline dan skenario what-if
- plausibility check sederhana dengan LOF terhadap reference data


In [1]:
from __future__ import annotations

import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.neighbors import LocalOutlierFactor
from IPython.display import display


def find_service_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "diabetify_cf").exists():
            return candidate
    raise RuntimeError("Could not find diabetify-cf service root from current working directory.")


SERVICE_ROOT = find_service_root(Path.cwd())
PROGRAM_ROOT = SERVICE_ROOT.parent

MODEL_PATH = PROGRAM_ROOT / "diabetify-ml" / "xg_model.pkl"
COLUMNS_PATH = PROGRAM_ROOT / "diabetify-ml" / "x_columns.pkl"
REFERENCE_DATA_PATH = SERVICE_ROOT / "artifacts" / "reference" / "reference_data.parquet"

with MODEL_PATH.open("rb") as f:
    model = pickle.load(f)

with COLUMNS_PATH.open("rb") as f:
    x_columns = list(pickle.load(f))

if REFERENCE_DATA_PATH.exists():
    reference_df = pd.read_parquet(REFERENCE_DATA_PATH)
    reference_df = reference_df[x_columns].copy()
else:
    reference_df = pd.DataFrame(columns=x_columns)

print("SERVICE_ROOT:", SERVICE_ROOT)
print("MODEL_PATH:", MODEL_PATH)
print("COLUMNS_PATH:", COLUMNS_PATH)
print("REFERENCE_DATA_PATH:", REFERENCE_DATA_PATH)
print("x_columns:", x_columns)
print("reference_df shape:", reference_df.shape)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_12984\1868992266.py:27: UserWarning: [19:53:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\data\../common/error_msg.h:82: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  model = pickle.load(f)


SERVICE_ROOT: d:\Perkuliahan\Tugas Akhir - The Last Chapter\Program\diabetify-cf
MODEL_PATH: d:\Perkuliahan\Tugas Akhir - The Last Chapter\Program\diabetify-ml\xg_model.pkl
COLUMNS_PATH: d:\Perkuliahan\Tugas Akhir - The Last Chapter\Program\diabetify-ml\x_columns.pkl
REFERENCE_DATA_PATH: d:\Perkuliahan\Tugas Akhir - The Last Chapter\Program\diabetify-cf\artifacts\reference\reference_data.parquet
x_columns: ['age', 'smoking_status', 'is_cholesterol', 'is_macrosomic_baby', 'moderate_physical_activity_frequency', 'is_bloodline', 'brinkman_index', 'BMI', 'is_hypertension']
reference_df shape: (2954, 9)


In [ ]:
display(pd.DataFrame({"feature": x_columns}))

if not reference_df.empty:
    display(reference_df.head(5))
    display(reference_df.describe().T)
else:
    print("reference_data.parquet belum tersedia atau kosong.")


In [ ]:
def make_feature_frame(rows: dict | list[dict] | pd.DataFrame) -> pd.DataFrame:
    if isinstance(rows, pd.DataFrame):
        frame = rows.copy()
    elif isinstance(rows, dict):
        frame = pd.DataFrame([rows])
    else:
        frame = pd.DataFrame(rows)

    missing = [col for col in x_columns if col not in frame.columns]
    extra = [col for col in frame.columns if col not in x_columns]

    if missing:
        raise ValueError(f"Missing feature(s): {missing}")
    if extra:
        raise ValueError(f"Unknown feature(s): {extra}")

    frame = frame[x_columns].copy()
    for col in x_columns:
        frame[col] = pd.to_numeric(frame[col], errors="raise")
    return frame


def predict_frame(rows: dict | list[dict] | pd.DataFrame) -> pd.DataFrame:
    frame = make_feature_frame(rows)
    probabilities = model.predict_proba(frame)
    result = frame.copy()
    result["proba_low_risk"] = probabilities[:, 0]
    result["proba_high_risk"] = probabilities[:, 1]
    result["predicted_class"] = np.where(result["proba_high_risk"] >= 0.5, "high_risk", "low_risk")
    return result


def fit_lof(reference_frame: pd.DataFrame) -> LocalOutlierFactor | None:
    if reference_frame.empty or len(reference_frame) < 2:
        return None
    lof = LocalOutlierFactor(n_neighbors=min(20, len(reference_frame) - 1), novelty=True)
    lof.fit(reference_frame[x_columns])
    return lof


lof_model = fit_lof(reference_df)


def add_plausibility(result: pd.DataFrame) -> pd.DataFrame:
    if lof_model is None:
        enriched = result.copy()
        enriched["lof_decision_function"] = np.nan
        enriched["plausible_by_lof"] = pd.NA
        return enriched

    feature_frame = make_feature_frame(result[x_columns])
    enriched = result.copy()
    enriched["lof_decision_function"] = lof_model.decision_function(feature_frame)
    enriched["plausible_by_lof"] = lof_model.predict(feature_frame) == 1
    return enriched


def compare_scenarios(baseline: dict, scenarios: dict[str, dict]) -> pd.DataFrame:
    rows = []
    rows.append({"scenario": "baseline", **baseline})
    for scenario_name, updates in scenarios.items():
        candidate = dict(baseline)
        candidate.update(updates)
        rows.append({"scenario": scenario_name, **candidate})

    frame = pd.DataFrame(rows)
    labels = frame.pop("scenario")
    result = predict_frame(frame)
    result.insert(0, "scenario", labels)
    result = add_plausibility(result)

    baseline_high_risk = float(result.loc[result["scenario"] == "baseline", "proba_high_risk"].iloc[0])
    result["delta_high_risk_vs_baseline"] = result["proba_high_risk"] - baseline_high_risk
    return result.sort_values("proba_high_risk", ascending=False)


## Baseline Prediction

Ubah nilai di bawah ini sesuai profil yang ingin kamu uji. Nilai fitur harus mengikuti skema model persis seperti pada `x_columns.pkl`.


In [ ]:
baseline = {
    "age": 52,
    "smoking_status": 2,
    "is_cholesterol": 1,
    "is_macrosomic_baby": 2,
    "moderate_physical_activity_frequency": 0,
    "is_bloodline": 1,
    "brinkman_index": 2,
    "BMI": 31.2,
    "is_hypertension": 1,
}

baseline_result = add_plausibility(predict_frame(baseline))
display(baseline_result)


## What-If Scenarios

Patch hanya fitur yang ingin kamu ubah. Notebook akan merge patch tersebut ke baseline lalu menjalankan prediksi ulang.


In [ ]:
scenarios = {
    "more_activity": {
        "moderate_physical_activity_frequency": 5,
    },
    "lower_bmi": {
        "BMI": 26.5,
    },
    "quit_smoking": {
        "smoking_status": 1,
        "brinkman_index": 1,
    },
    "combined_improvement": {
        "moderate_physical_activity_frequency": 5,
        "BMI": 26.5,
        "smoking_status": 1,
        "brinkman_index": 1,
    },
}

scenario_result = compare_scenarios(baseline, scenarios)
display(scenario_result)


## Optional: Start From Reference Data Sample

Kalau ingin memulai eksperimen dari contoh nyata pada reference data, ambil satu baris lalu edit sebagai baseline baru.


In [ ]:
if reference_df.empty:
    print("reference_df kosong, lewati cell ini.")
else:
    sampled_baseline = reference_df.sample(1, random_state=42).iloc[0].to_dict()
    display(pd.DataFrame([sampled_baseline]))
    display(add_plausibility(predict_frame(sampled_baseline)))
